# Week 2 Hands-on exercise

The following hands-on exercise is a simplified replication of the paper by Krueger (1999), Experimental Estimates of Education Production Functions. The paper provides an econometric analysis of a randomized experiment conducted in Tennessee, US, the Student/Teacher Achievement Ratio experiment, or project STAR. The project was designed to answer the following research question: does class size affect the academic performance of students from kindergarten through third grade? Answering this type of question may be important to guide policymakers about which part of education, how, and how much money they should invest to strengthen the elementary education. The project was conducted by academic researchers in universities at Tennessee over four years from 1985-1986 school year. The sample of experiment includes approximately 11,600 students in 80 schools. Within each school, students and teachers were randomly assigned to either small classes (13 - 17 students per teacher), regular classes (22 - 25 students per teacher), or regular classes with aide (22 - 25 students per teacher including a full-time teacher's aide).

(link: http://piketty.pse.ens.fr/files/Krueger1999.pdf)

#### Q1. Write down the research question above. 

*Your answer:*

#### Q2. Import necessary packages such as numpy, pandas. The data set is given below. (You should be able to download the data set for your project on your own by searching it or asking it to LLMs.)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

star = sm.datasets.get_rdataset("STAR", "AER").data   # downloads the data: needs an internet connection
print(star.shape)    # (11598, 47)

#### Q3. Take a look at the data set. What variables do you see? Use the following command to see the data set for the first five observations. 

In [ ]:
print("The first five observations of the data set: ", star.head()) 
print("Summary of the data set: ", star.info())
print("Variable names: ", star.columns)

#### Q4. Describe the variables using summary statistics (mean/ median/ standard deviation..)

In [ ]:
# Your code here

In [ ]:
# Your code here

#### Q5. We are going to use the following variables. 'stark', 'star1', 'star2', 'star3','readk', 'read1', 'read2', 'read3', 'mathk', 'math1', 'math2', and 'math3'. Please read the summary page of variables in the data set. (link: https://vincentarelbundock.github.io/Rdatasets/doc/AER/STAR.html)

*Your answer:*

#### Q6. One of the practical issues you definitely face is how to handle missing values. They are often described as "NaN," the abbreviation for "Not a Number," or "NA", "Not Available." Here, we are going to keep the observations from the data set only if the observations have values on the variables we use. That is, if any observations containing "NaN" or "NA", we are going to eliminate the observations. Please see the below and how the original data set changes as you drop out all the missing values. 

How many missing values exist in each variable? How to check it? 

In [ ]:
# Your code here

How to create a data set for kindergarten without any missing values in the variables we use? There are two ways.

1. Use True/False and logical syntax. 

In [ ]:
# Your code here

2. Use `dropna` after the name of data set. 

In [ ]:
# Your code here

#### Q7. After we eliminate all the missing values to correctly estimate the causal parameters of interest using the data set for kindergarten, we create variables. 

1. Define the criterion to distinguish the treatment from the control in the project STAR. What status can be treated as "treat?" 

*Your answer:*

2. Create two groups, treated_k and control_k that contain observations corresponding to each treatment arm. 

In [ ]:
# Your code here

3. Compute the sample mean of outcome values in each group. Take the difference between them. What is the result? 

In [ ]:
# Your code here

#### Q8. How precise is the estimate? Standard error and 95% confidence interval

The estimate `diff` comes from one random assignment. If students had been assigned differently, we would get a different number. The **standard error** measures how much the estimate would move from one assignment to another.

Let $Y_i$ be the outcome of student $i$ (the average of the reading and math scores), $n_1$ and $n_0$ the numbers of students in small and regular classes, and $\overline{Y}_1$ and $\overline{Y}_0$ their mean outcomes.

**1. Sample variances within each group.** How spread out are the individual outcomes in each group?

$$
s_1^2 = \frac{1}{n_1 - 1} \sum_{i \in \text{small}} (Y_i - \overline{Y}_1)^2, \qquad
s_0^2 = \frac{1}{n_0 - 1} \sum_{i \in \text{regular}} (Y_i - \overline{Y}_0)^2
$$

Use `y_1`, `y_0`, `y1_bar_k`, `y0_bar_k`, `n_1` and `n_0` from Q7. Call the results `s2_1` and `s2_0`.

In [ ]:
# Your code here

**2. Standard error.** Dividing each variance by its group size gives the estimated variance of each group **mean**. The two means come from different students, so the variance of their difference is the sum:

$$
\text{SE} = \sqrt{\frac{s_1^2}{n_1} + \frac{s_0^2}{n_0}}
$$

Call the result `se`. (Hint: `np.sqrt`.)

In [ ]:
# Your code here

**3. 95% confidence interval.**

$$
\text{CI}_{95} = \left[\,\hat{\tau} - C_{0.975}\,\text{SE},\;\; \hat{\tau} + C_{0.975}\,\text{SE}\,\right],
$$

where $\hat{\tau} = \overline{Y}_1 - \overline{Y}_0$ is `diff`, and $C_{0.975} \approx 1.96$ is the 97.5% quantile of the standard normal distribution. The cell below computes $C_{0.975}$. Compute `ci_low` and `ci_high` and print all the numbers.

In [ ]:
from scipy.stats import norm

quantile_975 = norm.ppf(0.975)   # C_0.975

In [ ]:
# Your code here

#### Q9. The same steps in a few lines of `pandas`

**Your turn.** Read the section "In short, we can do all the previous code work as below" on the Week 2 page (<https://yoonismo.github.io/causal-computing/week2.html>). Compute the difference in means and the standard error again, this time with the `pandas` methods `.mean()` and `.var()`. Do you get the same numbers as above?

In [ ]:
# Your code here

#### Q10. The same numbers in one line

**Your turn.** Read the section "The same numbers in one line" on the Week 2 page. A regression of the outcome on a treatment indicator, with `cov_type="HC2"`, gives the same estimate and standard error.

First we need a data set with only small and regular classes, an indicator `small` (1 for a small class, 0 for a regular class) and the outcome `score`. The cell below builds it.

In [ ]:
kr = kinder[kinder["stark"].isin(["small", "regular"])].assign(
    small=lambda d: (d["stark"] == "small").astype(int),       # treatment indicator
    score=lambda d: (d["readk"] + d["mathk"]) / 2,             # outcome
)
kr[["stark", "small", "readk", "mathk", "score"]].head()

Now run the regression of `score` on `small` and print the coefficient on `small`, its standard error and the 95% interval.

In [ ]:
# Your code here

#### Q11. Interpret the result

In words a parent would understand: how much higher is the average score in small classes, and how sure are we? To judge the size, compare the difference with the standard deviation of the outcome in regular classes, `y_0.std()`.

In [ ]:
# Your code here

*Your answer:*

**Note.** Our standard error treats every student as independent. Krueger (1999) allows students in the same class to be correlated, so his standard errors are larger. Our data identify schools but not classes, so we cannot do the same.